<a href="https://colab.research.google.com/github/ekBlaise/huggingface_solution_measuring_time/blob/master/huggingface_solution_measuring_time.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip -q install datasets accelerate transformers==4.56.2 trl==0.22.2 peft bitsandbytes

In [10]:
import time
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

In [11]:
assert torch.cuda.is_available() # print if there is a GPT available
print("GPU:", torch.cuda.get_device_name())

GPU: Tesla T4


In [12]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATASET_SIZE = 2000
MAX_SEQ_LENGTH = 1024
MAX_STEPS = 30
BATCH_SIZE = 2
GRAD_ACCUM = 4
LR = 2e-4

In [14]:
raw_dataset = load_dataset("yahma/alpaca-cleaned", split="train").select(range(DATASET_SIZE))
raw_dataset.column_names

['output', 'input', 'instruction']

In [13]:
raw_dataset = load_dataset("yahma/alpaca-cleaned", split="train").select(range(DATASET_SIZE))

alpaca_prompt = """Below is an instruction.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}
"""

def convert_to_text(example):
    return {
        "text": alpaca_prompt.format(
            instruction=example["instruction"],
            input=example["input"],
            output=example["output"],
        )
    }

dataset = raw_dataset.map(
    convert_to_text,
    remove_columns=raw_dataset.column_names,
)

print("Sample training text:\n")
print(dataset[0]["text"])

Sample training text:

Below is an instruction.

### Instruction:
Give three tips for staying healthy.

### Input:


### Response:
1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.

2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.

3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.



In [15]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [16]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [17]:
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [18]:
sft_config = SFTConfig(
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,

    output_dir="hf_out",
    report_to="none",

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,

    # Stable for Colab/T4-style GPUs
    fp16=False,
    bf16=False,

    optim="paged_adamw_8bit",
    logging_steps=5,
    save_strategy="no",
)

In [19]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=sft_config,
)


Adding EOS to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [20]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

start_time = time.time()

trainer.train()

torch.cuda.synchronize()

train_time = round(time.time() - start_time, 2)
train_vram_allocated = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
train_vram_reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

print("\nHF TRAINING RESULTS")
print("Train time/sec:", train_time)
print("Peak allocated VRAM/GB:", train_vram_allocated)
print("Peak reserved VRAM/GB:", train_vram_reserved)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
5,1.602000
10,1.640700
15,1.461300
20,1.615500
25,1.540100
30,1.553700



HF TRAINING RESULTS
Train time/sec: 71.52
Peak allocated VRAM/GB: 1.756
Peak reserved VRAM/GB: 2.199


In [21]:
model.eval()
model.config.use_cache = True

instruction = "Give three tips for staying healthy."

inference_prompt = """Below is an instruction.

### Instruction:
{instruction}

### Input:


### Response:
""".format(instruction=instruction)

inputs = tokenizer(inference_prompt, return_tensors="pt").to("cuda")

# Warm-up generation
with torch.inference_mode():
    _ = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

torch.cuda.synchronize()
t0 = time.time()

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

torch.cuda.synchronize()
t1 = time.time()

input_tokens = inputs["input_ids"].shape[-1]
generated_token_ids = output[0][input_tokens:]

generated_tokens = len(generated_token_ids)
tokens_per_sec = round(generated_tokens / (t1 - t0), 2)

answer = tokenizer.decode(generated_token_ids, skip_special_tokens=True)

print("\nHF FINAL RESULTS")
print("Train time/sec:", train_time)
print("Peak allocated VRAM/GB:", train_vram_allocated)
print("Peak reserved VRAM/GB:", train_vram_reserved)
print("Generated tokens/sec:", tokens_per_sec)

print("\nMODEL ANSWER:")
print(answer)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



HF FINAL RESULTS
Train time/sec: 71.52
Peak allocated VRAM/GB: 1.756
Peak reserved VRAM/GB: 2.199
Generated tokens/sec: 13.2

MODEL ANSWER:
1. Eat a balanced diet that includes a variety of fruits, vegetables, whole grains, and lean proteins.

2. Get enough sleep each night.

3. Stay hydrated by drinking plenty of water throughout the day.

4. Exercise regularly, but not too much.

5. Avoid smoking and excessive alcohol consumption.

6. Maintain a healthy weight by monitoring your body fat percentage.

7. Limit your intake of processed foods and sugary drinks.

8.
